In [1]:
## 1.Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp '/content/drive/My Drive/UAV/data1.zip' '/content/data1.zip'
!unzip '/content/data1.zip' -d '/content/'

In [ ]:
pip install -U albumentations

In [ ]:
!git clone https://github.com/ultralytics/ultralytics.git
!cd ultralytics && pip install .

In [ ]:
import yaml
from ultralytics import YOLO
from google.colab import files, drive
import os

# Mount Google Drive to save weights
drive.mount('/content/drive')

# Paths to the unzipped dataset
train_images_path = '/content/data1/images/train'
val_images_path = '/content/data1/images/val'
train_labels_path = '/content/data1/labels/train'
val_labels_path = '/content/data1/labels/val'

# Create a temporary dictionary for data configuration
data_config = {
    'train': train_images_path,  # Path to training images
    'val': val_images_path,      # Path to validation images
    'nc': 35,                    # Number of classes
    'names': [                   # Class names
        "Traffic Signal", "Lamp Post", "Zebra Crossing", "Bike", "Car", "Rikshaw", "Tyre Works",
        "Tree", "Tractor", "Cattle", "Vegetation", "Electricity Pole", "Building", "Board", "Wall",
        "Person", "Bus", "Bridge", "Road Divider", "Tempo", "Traffic Sign Board", "Flag", "Crane",
        "Cycle", "Dog", "Truck", "Glove", "Overbridge", "Manhole", "Bus Stop", "Barricade",
        "Petrol Pump", "Ambulance", "Goat", "Cart"
    ]
}

# Save the configuration to a temporary file
config_file_path = '/content/drive/MyDrive/UAV/config.yaml'
with open(config_file_path, 'w') as file:
    yaml.dump(data_config, file)

print(f"Config file saved at {config_file_path}")

# Upload the weights file 'last.pt' to continue training
print("Please upload the weights file.")
uploaded = files.upload()  # Upload the file

# Print the names of the uploaded files for debugging
print("Uploaded files:", uploaded.keys())

# Ensure the uploaded file is correctly handled
uploaded_files = list(uploaded.keys())
if len(uploaded_files) == 0:
    raise FileNotFoundError("No file uploaded. Please upload the correct weights file.")
uploaded_file_name = uploaded_files[0]

# Handle file names like 'last (8).pt', 'last (9).pt', etc.
if 'last.pt' not in uploaded_file_name:
    # Rename the uploaded file to 'last.pt' if it contains 'last' and a number
    new_file_name = 'last.pt'
    os.rename(f'/content/{uploaded_file_name}', f'/content/{new_file_name}')
    uploaded_file_name = new_file_name
    print(f"Renamed the uploaded file to: {uploaded_file_name}")

# Path to the uploaded weights file
last_weights_path = f'/content/{uploaded_file_name}'
print(f"Using uploaded file: {uploaded_file_name}")

# Load the model from the uploaded weights
model = YOLO(last_weights_path)

# Resume training for 5 epochs or until 50 epochs total
total_epochs = 50
current_epoch = model.ckpt['epoch'] + 1  # Add 1 to the epoch because YOLOv8 starts from 0
remaining_epochs = total_epochs - current_epoch
epochs_to_train = min(5, remaining_epochs)  # Train for 5 or fewer epochs if near the target

if remaining_epochs > 0:
    print(f"Training for {epochs_to_train} epochs (Current Epoch: {current_epoch}).")
    model.train(data=config_file_path, epochs=current_epoch + epochs_to_train, batch=5)

    # Update the current epoch after training
    current_epoch += epochs_to_train

    # Save the updated weights
    new_weights_path = f'/content/drive/MyDrive/updated_weights_epoch_{current_epoch}.pt'
    model.save(new_weights_path)
    print(f"Updated weights saved at {new_weights_path}.")
else:
    print("Training is already complete. Total 50 epochs reached!")

In [ ]:
import shutil
from google.colab import files

# Specify the source directory and the target ZIP file path
source_dir = "/content/runs"
output_zip = "/content/runs.zip"

# Create a ZIP file from the source directory
shutil.make_archive(output_zip.replace('.zip', ''), 'zip', source_dir)

# Download the ZIP file to the local system
files.download(output_zip)